# Deepfake Detection — Model Training Notebook

Trains a binary (real vs. fake) face classifier on the **140k Real and Fake Faces** dataset,
evaluates it, generates Grad-CAM explanations, and saves the checkpoint + metadata that the
Streamlit demo app (Phase 5, separate) will load.

Covers Phases 1–4 of the project plan:
1. Setup (dataset download)
2. Data preparation
3. Model development (transfer learning)
4. Evaluation and explainability (Grad-CAM)

**Prerequisites:** `pip install torch torchvision scikit-learn matplotlib seaborn opencv-python pillow tqdm kaggle`
(these should already be in `requirements.txt` per the project plan). Kaggle API credentials
(`kaggle.json`) must be in `~/.kaggle/` to auto-download the dataset — see https://www.kaggle.com/docs/api.


In [ ]:
import os

# ---- Config ----
# DATA_ROOT is set in the next cell, after kagglehub downloads the dataset.
SUBSET_PER_CLASS_TRAIN = 12000   # v1 uses a stratified subset, not the full 140k — see plan
SUBSET_PER_CLASS_VAL = 2000
SUBSET_PER_CLASS_TEST = 2000
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_EPOCHS = 8
LEARNING_RATE = 1e-4
SEED = 42
CHECKPOINT_DIR = "models/checkpoints"
FIGURES_DIR = "reports/figures"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

## Phase 1 — Setup: Get the Dataset (kagglehub, no manual download)

`kagglehub` downloads straight to the Colab VM — no manual browser download, no Drive upload. For a fully public dataset like this one it usually works with no credentials at all. If it ever raises an auth error, run `kagglehub.login()` in a new cell (pastes a Kaggle API key, no file setup) and re-run this cell.

In [ ]:
!pip install -q kagglehub

import kagglehub

kagglehub_path = kagglehub.dataset_download("xhlulu/140k-real-and-fake-faces")
print("Path to dataset files:", kagglehub_path)

DATA_ROOT = os.path.join(kagglehub_path, "real_vs_fake", "real-vs-fake")
print("DATA_ROOT:", DATA_ROOT)
assert os.path.isdir(DATA_ROOT), f"expected {DATA_ROOT} to exist — check kagglehub_path's contents with !find {kagglehub_path} -maxdepth 3"

In [ ]:
import random
import json
import copy

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from tqdm import tqdm

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## Phase 2 — Data Preparation

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


In [ ]:
def stratified_subset(dataset, per_class_count, seed=SEED):
    """Return a Subset with up to `per_class_count` samples per class, class-balanced."""
    targets = np.array(dataset.targets)
    indices = []
    rng = random.Random(seed)
    for class_idx in range(len(dataset.classes)):
        class_indices = np.where(targets == class_idx)[0].tolist()
        rng.shuffle(class_indices)
        indices.extend(class_indices[:per_class_count])
    rng.shuffle(indices)
    return Subset(dataset, indices)


train_dataset_full = datasets.ImageFolder(f"{DATA_ROOT}/train", transform=train_transform)
valid_dataset_full = datasets.ImageFolder(f"{DATA_ROOT}/valid", transform=eval_transform)
test_dataset_full = datasets.ImageFolder(f"{DATA_ROOT}/test", transform=eval_transform)

print("Classes:", train_dataset_full.classes)  # expect ['fake', 'real'] (alphabetical)

train_dataset = stratified_subset(train_dataset_full, SUBSET_PER_CLASS_TRAIN)
valid_dataset = stratified_subset(valid_dataset_full, SUBSET_PER_CLASS_VAL)
test_dataset = stratified_subset(test_dataset_full, SUBSET_PER_CLASS_TEST)

print(f"Train: {len(train_dataset)} | Valid: {len(valid_dataset)} | Test: {len(test_dataset)}")

class_names = train_dataset_full.classes  # index -> name
class_to_idx = train_dataset_full.class_to_idx


In [ ]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)


## Phase 3 — Model Development (Transfer Learning)

In [ ]:
def build_model(num_classes=2):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    return model


model = build_model(num_classes=len(class_names)).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)


In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, total_correct, total_samples = 0.0, 0, 0
    context = torch.enable_grad() if is_train else torch.no_grad()

    with context:
        for images, labels in tqdm(loader, leave=False):
            images, labels = images.to(device), labels.to(device)

            if is_train:
                optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            if is_train:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            total_correct += (preds == labels).sum().item()
            total_samples += images.size(0)

    return total_loss / total_samples, total_correct / total_samples


In [ ]:
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc = 0.0
best_model_state = None

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc = run_epoch(model, valid_loader, criterion, optimizer=None)
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch}/{NUM_EPOCHS} | "
          f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
          f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_state = copy.deepcopy(model.state_dict())

model.load_state_dict(best_model_state)
print(f"Best validation accuracy: {best_val_acc:.4f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history["train_acc"], label="train")
axes[1].plot(history["val_acc"], label="val")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/training_curves.png", dpi=150)
plt.show()


## Phase 4 — Evaluation and Explainability

In [ ]:
model.eval()
all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for images, labels in tqdm(test_loader, leave=False):
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        preds = outputs.argmax(dim=1).cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

acc = accuracy_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds)
rec = recall_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)

print(f"Test Accuracy:  {acc:.4f}")
print(f"Test Precision: {prec:.4f}")
print(f"Test Recall:    {rec:.4f}")
print(f"Test F1:        {f1:.4f}")
print()
print(classification_report(all_labels, all_preds, target_names=class_names))


In [ ]:
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix — Test Set")
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/confusion_matrix.png", dpi=150)
plt.show()


### Grad-CAM

Minimal hook-based Grad-CAM on the last conv block (`layer4`) of ResNet18 — no external
grad-cam package dependency, keeps the pipeline self-contained.

In [ ]:
class GradCAM:
    """Minimal Grad-CAM via forward/backward hooks on a target conv layer."""

    def __init__(self, model, target_layer):
        self.model = model
        self.gradients = None
        self.activations = None

        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, input_tensor, class_idx=None):
        self.model.zero_grad()
        output = self.model(input_tensor)

        if class_idx is None:
            class_idx = output.argmax(dim=1).item()

        score = output[0, class_idx]
        score.backward()

        gradients = self.gradients[0]          # (C, H, W)
        activations = self.activations[0]      # (C, H, W)

        weights = gradients.mean(dim=(1, 2))   # (C,)
        cam = torch.relu((weights[:, None, None] * activations).sum(dim=0))

        cam = cam.cpu().numpy()
        cam = cv2.resize(cam, (IMG_SIZE, IMG_SIZE))
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

        return cam, class_idx


gradcam = GradCAM(model, target_layer=model.layer4[-1])


In [ ]:
def denormalize(tensor):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    img = tensor.cpu() * std + mean
    img = img.clamp(0, 1).permute(1, 2, 0).numpy()
    return img


def overlay_heatmap(image, cam, alpha=0.4):
    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB) / 255.0
    overlay = (1 - alpha) * image + alpha * heatmap
    return np.clip(overlay, 0, 1)


def show_gradcam_examples(model, dataset, class_names, n=4, only_correct=None, seed=SEED):
    """Display n Grad-CAM examples. only_correct=True/False filters for correct/incorrect
    predictions; None shows a mix."""
    rng = random.Random(seed)
    order = list(range(len(dataset)))
    rng.shuffle(order)

    shown = 0
    fig, axes = plt.subplots(1, n, figsize=(3 * n, 3.5))
    if n == 1:
        axes = [axes]

    for idx in order:
        if shown >= n:
            break

        image_tensor, true_label = dataset[idx]
        input_tensor = image_tensor.unsqueeze(0).to(device)

        cam, pred_label = gradcam.generate(input_tensor)
        is_correct = (pred_label == true_label)

        if only_correct is not None and is_correct != only_correct:
            continue

        img_np = denormalize(image_tensor)
        overlay = overlay_heatmap(img_np, cam)

        axes[shown].imshow(overlay)
        axes[shown].set_title(
            f"true={class_names[true_label]}\npred={class_names[pred_label]}",
            fontsize=9,
            color="green" if is_correct else "red",
        )
        axes[shown].axis("off")
        shown += 1

    plt.tight_layout()
    return fig


fig_correct = show_gradcam_examples(model, test_dataset, class_names, n=4, only_correct=True)
fig_correct.savefig(f"{FIGURES_DIR}/gradcam_correct_examples.png", dpi=150)
plt.show()

fig_incorrect = show_gradcam_examples(model, test_dataset, class_names, n=4, only_correct=False)
fig_incorrect.savefig(f"{FIGURES_DIR}/gradcam_incorrect_examples.png", dpi=150)
plt.show()


## Save Artifacts for the Streamlit App (Phase 5)

The Streamlit demo (built separately) loads `deepfake_resnet18.pt` + `metadata.json` from
`models/checkpoints/` — this cell writes both.

In [ ]:
checkpoint_path = f"{CHECKPOINT_DIR}/deepfake_resnet18.pt"
torch.save(model.state_dict(), checkpoint_path)

metadata = {
    "architecture": "resnet18",
    "num_classes": len(class_names),
    "class_names": class_names,          # index -> name, e.g. ['fake', 'real']
    "class_to_idx": class_to_idx,
    "img_size": IMG_SIZE,
    "normalize_mean": IMAGENET_MEAN,
    "normalize_std": IMAGENET_STD,
    "test_accuracy": acc,
    "test_precision": prec,
    "test_recall": rec,
    "test_f1": f1,
}

with open(f"{CHECKPOINT_DIR}/metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Saved checkpoint to {checkpoint_path}")
print(f"Saved metadata to {CHECKPOINT_DIR}/metadata.json")


## Bring the results back to your local machine

Run the cell below to zip the checkpoint + figures and download them through the browser. Unzip into the same paths (`models/checkpoints/`, `reports/figures/`) in your local clone of the repo, then the Streamlit demo can load them.

In [ ]:
import shutil
shutil.make_archive("/content/checkpoint_result", "zip", ".", CHECKPOINT_DIR)
shutil.make_archive("/content/figures_result", "zip", ".", FIGURES_DIR)

from google.colab import files
files.download("/content/checkpoint_result.zip")
files.download("/content/figures_result.zip")

## Limitations (for the report)

- **Cross-GAN generalization**: this model is trained only on StyleGAN-generated fakes
  (the 140k dataset's fake half). It may not reliably flag deepfakes produced by other
  generators (diffusion models, other GAN architectures) — call this out explicitly rather
  than overstating real-world robustness.
- **Static images only** — this pipeline does not detect video-based/temporal deepfakes.
- Subset training (not the full 140k) trades some accuracy ceiling for time budget; note the
  actual subset sizes used when reporting results.
